# Project Pipeline — ETF Strategy Robustness Evaluation

**Name:** Jesse Wang
**Date:** 2026-08-20

The master pipeline, extended every stage. Currently covers:

- **Stage 04** — acquire ETF prices (yfinance) + S&P 500 constituents (Wikipedia)
- **Stage 05** — store raw data reproducibly, save/reload Parquet with validation
- **Stage 06** — preprocess the price panel into a modeling-ready return series
- **Stage 07** — detect & flag outliers (per ticker), sensitivity-check the impact

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys
if Path.cwd().name == 'notebooks':
    os.chdir('..')                    # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))     # so `from src....` imports work
print('working from:', ROOT.name)

working from: project


## Stage 04 — Data Acquisition & Ingestion

Acquire data programmatically (API pull + permitted public table), validate it, and save it to `data/raw/`.

In [2]:
# --- imports, env, paths ---
import requests
import pandas as pd
from bs4 import BeautifulSoup
import yfinance as yf

from src import io
from src.config import load_env

load_env()              # read .env (API key + data dirs) into os.environ
RAW, PROC = io.get_paths()
print('RAW  ->', RAW.resolve())
print('PROC ->', PROC.resolve())

def validate_acquisition(df, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape,
            'na_total': int(df.isna().sum().sum()),
            'dtypes': {c: str(df[c].dtype) for c in required if c in df.columns}}

RAW  -> /Users/wangjing/bootcamp_Jesse_Wang/project/data/raw
PROC -> /Users/wangjing/bootcamp_Jesse_Wang/project/data/processed


### Part 1 — API pull: ETF daily prices

Universe: `SPY`, `QQQ`, `IWM`, `GLD`, `TLT` — a representative cross-asset rotation basket. Uses Alpha Vantage if a key is set in `.env`, otherwise falls back to `yfinance`.

In [3]:
# --- Part 1: API pull ---
ETFS = ['SPY', 'QQQ', 'IWM', 'GLD', 'TLT']
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
print('Using Alpha Vantage:', USE_ALPHA)

def fetch_alpha(sym: str) -> pd.DataFrame:
    url = 'https://www.alphavantage.co/query'
    params = {'function': 'TIME_SERIES_DAILY', 'symbol': sym,
              'outputsize': 'compact', 'apikey': os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k]
    if not key:
        raise RuntimeError(f'Alpha Vantage returned no series for {sym} (rate limit / premium field?)')
    out = (pd.DataFrame(js[key[0]]).T.rename_axis('date').reset_index()
           [['date', '4. close']].rename(columns={'4. close': 'close'}))
    out['date'] = pd.to_datetime(out['date'])
    out['close'] = pd.to_numeric(out['close'])
    return out

frames = []
for sym in ETFS:
    d = None
    if USE_ALPHA:
        try:
            d = fetch_alpha(sym)
        except Exception as e:
            print(f'Alpha Vantage failed for {sym}, falling back to yfinance: {e}')
    if d is None:
        d = yf.download(sym, period='1y', interval='1d', auto_adjust=False,
                        progress=False, multi_level_index=False)
        d = d.reset_index()[['Date', 'Close']]
        d.columns = ['date', 'close']
    d['ticker'] = sym
    frames.append(d)

df_api = pd.concat(frames, ignore_index=True)
df_api['date'] = pd.to_datetime(df_api['date'])
df_api = df_api.sort_values(['ticker', 'date']).reset_index(drop=True)
df_api = df_api[['date', 'ticker', 'close']]

v_api = validate_acquisition(df_api, ['date', 'ticker', 'close'])
print('validation:', v_api)

etf_path = RAW / f"api_source-yfinance_etfs_{io.ts()}.csv"
io.write_df(df_api, etf_path)
print('Saved ->', etf_path)

Using Alpha Vantage: False


validation: {'missing': [], 'shape': (1255, 3), 'na_total': 0, 'dtypes': {'date': 'datetime64[ns]', 'ticker': 'object', 'close': 'float64'}}
Saved -> data/raw/api_source-yfinance_etfs_20260820-160228.csv


### Part 2 — Scrape a permitted public table

S&P 500 constituents from Wikipedia's `table#constituents`.

In [4]:
# --- Part 2: scrape ---
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'ETF-Strategy-Project/1.0'}

resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.find('table', id='constituents') or soup.find('table', class_='wikitable')
if table is None:
    raise RuntimeError('no <table> found on the page')
rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
        for tr in table.find_all('tr')]
header, *data = [r for r in rows if r]
df_scrape = pd.DataFrame(data, columns=header)
if 'CIK' in df_scrape.columns:
    df_scrape['CIK'] = pd.to_numeric(df_scrape['CIK'], errors='coerce').astype('Int64')
if 'Date added' in df_scrape.columns:
    df_scrape['Date added'] = pd.to_datetime(df_scrape['Date added'], errors='coerce')

v_scrape = validate_acquisition(df_scrape, list(df_scrape.columns))
print('validation:', v_scrape)

sp500_path = RAW / f"scrape_site-wikipedia_table-sp500_{io.ts()}.csv"
io.write_df(df_scrape, sp500_path)
print('Saved ->', sp500_path)

validation: {'missing': [], 'shape': (503, 8), 'na_total': 0, 'dtypes': {'Symbol': 'object', 'Security': 'object', 'GICSSector': 'object', 'GICS Sub-Industry': 'object', 'Headquarters Location': 'object', 'Date added': 'datetime64[ns]', 'CIK': 'Int64', 'Founded': 'object'}}
Saved -> data/raw/scrape_site-wikipedia_table-sp500_20260820-160228.csv


### Acquisition documentation

- **API source:** Alpha Vantage `TIME_SERIES_DAILY` (fallback `yfinance.download`), raw `close`, daily bars, ~1y history per ticker.
- **Scrape source:** `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies`, `table#constituents`; `CIK` parsed as integer, `Date added` as datetime.
- **Secrets:** key read from `.env` via `python-dotenv`, never hard-coded; `.env` is gitignored.
- **Reproducibility:** timestamped filenames so reruns never overwrite earlier snapshots.

## Stage 05 — Data Storage

Save/reload the raw panel through `src/io.py`; CSV for `data/raw/`, Parquet for `data/processed/` (dtype-preserving), with reload validation.

In [5]:
# --- Stage 05: raw CSV -> processed Parquet, reload + validate ---
etf_raw = io.read_df(etf_path)               # read_df re-parses `date` on CSV load
etf_processed_path = PROC / f"etf_prices_{io.ts()}.parquet"
io.write_df(etf_raw, etf_processed_path)     # Parquet preserves dtypes

reloaded = io.read_df(etf_processed_path)
print('validation:', io.validate_loaded(etf_raw, reloaded))
print('\nparquet dtypes (dtype-preserving):')
print(reloaded.dtypes)

validation: {'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'close_is_numeric': True}

parquet dtypes (dtype-preserving):
date      datetime64[ns]
ticker            object
close            float64
dtype: object


## Stage 06 — Data Preprocessing

Clean the price panel into a modeling-ready return series via `src/cleaning.py` + `src/utils.py`.

In [6]:
# --- Stage 06a: inspect missingness ---
prices = io.read_df(etf_path)
print('raw shape:', prices.shape)
print('missing by column:')
print(prices.isna().sum())

raw shape: (1255, 3)
missing by column:
date      0
ticker    0
close     0
dtype: int64


In [7]:
# --- Stage 06b: clean ---
from src import cleaning, utils

# 1) impute any missing close with the ticker median (robust to outliers/skew)
cleaned = cleaning.fill_missing_median(prices, columns=['close'])

# 2) compute daily simple returns within each ticker
cleaned = utils.compute_returns(cleaned, price_col='close', group_col='ticker', method='simple')

# 3) drop rows with a missing return (each ticker's first day has no prior close)
cleaned = cleaning.drop_missing(cleaned, columns=['return'])

print('cleaned shape:', cleaned.shape)
print('missing after cleaning:', int(cleaned.isna().sum().sum()))
cleaned.head()

cleaned shape: (1250, 4)
missing after cleaning: 0


,date,ticker,close,return
1,2025-08-22,GLD,310.579987,0.010706
2,2025-08-25,GLD,309.829987,-0.002415
3,2025-08-26,GLD,312.079987,0.007262
4,2025-08-27,GLD,312.709991,0.002019
5,2025-08-28,GLD,315.029999,0.007419


In [8]:
# --- Stage 06c: before/after missing comparison ---
before = prices.isna().sum()
after = cleaned.isna().sum()
pd.DataFrame({'before': before, 'after': after}).fillna(0).astype(int)

,before,after
close,0,0
date,0,0
return,0,0
ticker,0,0


In [9]:
# --- Stage 06d: save cleaned dataset + verify ---
cleaned_path = PROC / f"etf_returns_cleaned_{io.ts()}.csv"
io.write_df(cleaned, cleaned_path)

reloaded = io.read_df(cleaned_path)
assert reloaded.shape == cleaned.shape, 'shape mismatch on reload'
assert not reloaded['return'].isna().any(), 'cleaned data still has missing returns'
print('Saved ->', cleaned_path)
print('Reload OK:', reloaded.shape, '| no missing returns')

Saved -> data/processed/etf_returns_cleaned_20260820-160228.csv
Reload OK: (1250, 4) | no missing returns


## Stage 07 — Outlier Analysis

Detect outliers in the cleaned daily returns **within each ticker**, flag them
(never auto-delete), and save a flagged + winsorized copy for the modeling stage.
Full write-up in `docs/outliers.md`; sensitivity comparison in
`notebooks/sensitivity_outliers.ipynb`.

In [10]:
# --- Stage 07: flag outliers per ticker (keep, don't delete) ---
from src import outliers

cleaned['outlier_z'] = outliers.flag_outliers(
    cleaned, value_col='return', group_col='ticker', method='zscore', threshold=3.0)

print('Z-score (threshold=3) outlier summary:')
print(outliers.summarize_outliers(cleaned, method='zscore'))

# winsorized copy (5%/95% per ticker) for the modeling stage
cleaned['return_winsorized'] = outliers.winsorize_outliers(
    cleaned, value_col='return', group_col='ticker')

flagged_path = PROC / f"etf_returns_flagged_{io.ts()}.csv"
io.write_df(cleaned, flagged_path)
print('Saved ->', flagged_path.name)

# the flagged days — genuine tail events, not errors
cleaned.loc[cleaned['outlier_z'], ['date', 'ticker', 'return']].sort_values('return')

Z-score (threshold=3) outlier summary:
       flagged  fraction
GLD          3    0.0120
IWM          2    0.0080
QQQ          1    0.0040
SPY          4    0.0160
TLT          1    0.0040
total       11    0.0088
Saved -> etf_returns_flagged_20260820-160228.csv


,date,ticker,return
111,2026-01-30,GLD,-0.102742
42,2025-10-21,GLD,-0.064269
700,2026-06-05,QQQ,-0.048001
449,2026-06-05,IWM,-0.035478
788,2025-10-10,SPY,-0.027028
951,2026-06-05,SPY,-0.025809
1149,2026-03-20,TLT,-0.018974
910,2026-04-08,SPY,0.025470
905,2026-03-31,SPY,0.029068
252,2025-08-22,IWM,0.039209


## Assumptions & tradeoffs

- **Median imputation** (`fill_missing_median`) assumes missing closes are MCAR/MAR; median is robust to outliers/skew.
- **Returns within ticker** (`compute_returns`) — the first row of each ticker is dropped (`drop_missing` on `return`) because there is no prior close.
- **No normalization of raw prices** — rescaling `close` would destroy cross-ticker comparability and price levels are non-stationary; `normalize_data` is reserved for return/feature rescaling in the modeling stage.
- **Daily public data only**; intraday effects are not modeled.
- **Outliers are flagged, not deleted** (Z-score `threshold=3` within each ticker); the tail days are genuine events, so deletion/capping are explicit, optional treatments (see `docs/outliers.md`).
